# Calibration Plots

Inspect COMAP calibrator observations (Jupiter, CygA, CasA, TauA).

**Outputs**
1. Time series of each calibrator with bad fits cut and a best-fit model overlaid (polynomial or median-filter trend).
2. Mean calibration factor per feed and band, shown as a table for each calibrator.
3. Histograms of calibration factors split into user-defined date ranges (dd/mm/yyyy).

Configure paths and cuts in the **Configuration** cell, then run top-to-bottom.

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import h5py
from tqdm import tqdm
from astropy.time import Time
from scipy.ndimage import median_filter

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

sys.path.append('../')
from modules.SQLModule.SQLModule import db
from modules.utils import CaliModels

plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 150,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
})

In [ ]:
# --- Configuration ----------------------------------------------------------

DB_PATH      = '../databases/COMAP_manchester.db'
CACHE_DIR    = Path('source_fitting')
CACHE_DIR.mkdir(exist_ok=True)
REBUILD_CACHE = False       # set True to re-scan all level2 files

SOURCES = ['jupiter', 'CygA', 'CasA', 'TauA']

# Instrument layout
N_FEEDS    = 19
BAND_FREQS = np.array([26.5, 27.5, 28.5, 29.5, 30.5, 31.5, 32.5, 33.5])  # GHz
N_BANDS    = 4
N_CHANS    = 2  # per band  -> 8 total frequency channels arranged (band, chan)

# Quality cuts applied per (obs, feed, band, channel) fit
QUALITY_CUTS = dict(
    chi2_max         = 300.0,   # reduced chi^2 upper limit
    max_offset_arcmin= 1.0,     # |RA|, |Dec| pointing offset cut
    min_flux         = 0.0,     # measured Jy lower limit
)

# Time-series model
MODEL_KIND      = 'median'      # 'median' or 'poly'
MEDIAN_KERNEL   = 21            # samples (after time-sorting) for median filter
POLY_DEGREE     = 3

# Histogram date ranges (dd/mm/yyyy)
DATE_RANGES = [
    ('01/01/2019', '31/12/2020'),
    ('01/01/2021', '31/12/2022'),
    ('01/01/2023', '31/12/2024'),
]

# Flux model classes available for each source
FLUX_MODELS = {
    'jupiter': CaliModels.JupiterFluxModel(),
    'CygA'   : CaliModels.CygAFlux,            # function-style
    'CasA'   : CaliModels.CasAFluxModel(),
    'TauA'   : CaliModels.TauAFluxModel(),
}

def _model_flux(source, nu_ghz, mjd):
    """Return expected flux (Jy) at a given frequency and time."""
    m = FLUX_MODELS[source]
    if callable(m) and not isinstance(m, type):
        return m(nu_ghz, mjd)
    return m(nu_ghz, mjd)

In [ ]:
# --- Load calibrator fits (with on-disk cache) ------------------------------

FIT_KEYS = ['amplitudes', 'amplitude_errors', 'chi2',
            'ra_offset', 'dec_offset',
            'major_std', 'minor_std', 'flux']

def _load_source(source):
    """Scan all level2 files for one calibrator and return arrays."""
    db.connect(DB_PATH)
    cal_list = db.query_source_group_list('Calibrator', source, return_dict=False)

    rows = {k: [] for k in FIT_KEYS}
    obsids, times = [], []

    for obsid, c in tqdm(cal_list.items(), desc=source, leave=False):
        if c.level2_path is None:
            continue
        try:
            with h5py.File(c.level2_path, 'r') as h:
                grp = h.get('level2/calibrator_fit')
                if grp is None:
                    continue
                for k in FIT_KEYS:
                    rows[k].append(grp[k][...])
            t = Time(datetime.strptime(c.utc_start, '%Y-%m-%d-%H:%M:%S')).mjd
            obsids.append(obsid)
            times.append(t)
        except (OSError, KeyError):
            continue

    out = {k: np.array(v) for k, v in rows.items()}
    out['obsids'] = np.array(obsids)
    out['times']  = np.array(times)
    return out


def load_calibrators(sources=SOURCES, rebuild=REBUILD_CACHE):
    data = {}
    for src in sources:
        cache = CACHE_DIR / f'{src}_source_fits.npy'
        if cache.exists() and not rebuild:
            data[src] = np.load(cache, allow_pickle=True).item()
        else:
            data[src] = _load_source(src)
            np.save(cache, data[src])
        print(f'  {src:<8s}  n_obs = {len(data[src]["times"])}')
    return data


cal_data = load_calibrators()

In [ ]:
# --- Quality mask & calibration factor --------------------------------------

def quality_mask(d, cuts=QUALITY_CUTS):
    """Return boolean mask (n_obs, n_feeds, n_bands, n_chans) of good fits."""
    ra_arcmin  = np.abs(d['ra_offset'])  * 60.0
    dec_arcmin = np.abs(d['dec_offset']) * 60.0
    return (
        (d['chi2']  < cuts['chi2_max']) &
        (ra_arcmin  < cuts['max_offset_arcmin']) &
        (dec_arcmin < cuts['max_offset_arcmin']) &
        (d['flux']  > cuts['min_flux']) &
        np.isfinite(d['flux'])
    )


def calibration_factor(source, d):
    """
    Cal factor = model_flux(nu, mjd) / measured_flux.
    Returns array shaped (n_obs, n_feeds, n_bands, n_chans).
    """
    flux_meas = d['flux']                                  # (n_obs, F, B, C)
    times     = d['times']                                 # (n_obs,)
    nu        = BAND_FREQS.reshape(N_BANDS, N_CHANS)       # (B, C)

    expected = np.empty_like(flux_meas)
    for b in range(N_BANDS):
        for c in range(N_CHANS):
            expected[:, :, b, c] = _model_flux(source, nu[b, c], times)[:, None]

    with np.errstate(divide='ignore', invalid='ignore'):
        factor = expected / flux_meas
    factor[~np.isfinite(factor)] = np.nan
    return factor


def trend_model(t, y, kind=MODEL_KIND, kernel=MEDIAN_KERNEL, degree=POLY_DEGREE):
    """Fit a smooth model to a (t, y) timeseries; returns y_model on the same t."""
    order = np.argsort(t)
    t_s, y_s = t[order], y[order]
    if kind == 'poly':
        p = np.polyfit(t_s, y_s, degree)
        y_fit = np.polyval(p, t_s)
    elif kind == 'median':
        k = max(3, kernel | 1)   # ensure odd
        y_fit = median_filter(y_s, size=min(k, len(y_s)))
    else:
        raise ValueError(f'Unknown model kind: {kind}')
    inv = np.empty_like(y_fit)
    inv[order] = y_fit
    return inv

In [ ]:
# --- Time series with best-fit trend ----------------------------------------

def _mjd_to_dt(mjd):
    return Time(mjd, format='mjd').to_datetime()

def plot_timeseries(source, feed=1, band=0, chan=0,
                    kind=MODEL_KIND, ax=None):
    d     = cal_data[source]
    mask  = quality_mask(d)
    flux  = d['flux'][:, feed - 1, band, chan]
    err   = d['amplitude_errors'][:, feed - 1, band, chan]
    t     = d['times']
    gd    = mask[:, feed - 1, band, chan]

    if gd.sum() < 3:
        print(f'{source} feed {feed} band {band} ch {chan}: too few good points')
        return

    tg, yg, eg = t[gd], flux[gd], err[gd]
    y_trend = trend_model(tg, yg, kind=kind)

    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 4))
    dts = [_mjd_to_dt(x) for x in tg]
    order = np.argsort(tg)

    ax.errorbar(dts, yg, yerr=eg, fmt='o', ms=3.5, alpha=0.55,
                color='#1f77b4', ecolor='#88aacc', label=f'measurements (N={gd.sum()})')
    ax.plot(np.array(dts)[order], y_trend[order], '-', lw=2,
            color='#d62728', label=f'{kind} trend')

    # expected model flux at the centre frequency
    nu = BAND_FREQS.reshape(N_BANDS, N_CHANS)[band, chan]
    model_curve = _model_flux(source, nu, tg)
    ax.plot(np.array(dts)[order], model_curve[order], '--', lw=1.5,
            color='k', alpha=0.6, label=f'expected ({nu:.1f} GHz)')

    ax.set_title(f'{source}  —  feed {feed}, band {band}, chan {chan}')
    ax.set_ylabel('Measured flux [Jy]')
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    ax.legend(loc='best')
    return ax


# Example: one timeseries panel per calibrator (feed 1, band 0, chan 0)
fig, axes = plt.subplots(len(SOURCES), 1, figsize=(11, 3.4 * len(SOURCES)),
                         sharex=False)
for src, ax in zip(SOURCES, np.atleast_1d(axes)):
    plot_timeseries(src, feed=1, band=0, chan=0, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# --- Mean calibration-factor tables -----------------------------------------

def cal_factor_table(source):
    """Mean calibration factor per (feed, band), averaged over channels and time."""
    d      = cal_data[source]
    mask   = quality_mask(d)
    factor = calibration_factor(source, d)
    factor = np.where(mask, factor, np.nan)              # (n_obs, F, B, C)
    mean_fb = np.nanmean(factor, axis=(0, 3))            # -> (F, B)

    df = pd.DataFrame(
        mean_fb,
        index=[f'feed {i+1:02d}' for i in range(N_FEEDS)],
        columns=[f'band {b}' for b in range(N_BANDS)],
    )
    df.index.name = source
    return df


for src in SOURCES:
    df = cal_factor_table(src)
    print(f'\nMean calibration factor (model/measured) — {src}')
    display(df.style.format('{:.3f}').background_gradient(cmap='RdBu_r', axis=None))

In [ ]:
# --- Calibration-factor histograms split by date range ----------------------

def _ddmmyyyy_to_mjd(s):
    return Time(datetime.strptime(s, '%d/%m/%Y')).mjd


def plot_cal_histograms(source, feeds=None, bands=None,
                        date_ranges=DATE_RANGES, bins=40):
    """
    Overlay histograms of the calibration factor for `source`, one per date range.

    feeds, bands : iterables (1-indexed feeds, 0-indexed bands) or None for all.
    """
    d      = cal_data[source]
    mask   = quality_mask(d)
    factor = calibration_factor(source, d)
    factor = np.where(mask, factor, np.nan)              # (n_obs, F, B, C)

    if feeds is not None:
        f_idx = [f - 1 for f in feeds]
        factor = factor[:, f_idx, :, :]
    if bands is not None:
        factor = factor[:, :, list(bands), :]

    mjd_ranges = [(_ddmmyyyy_to_mjd(a), _ddmmyyyy_to_mjd(b)) for a, b in date_ranges]
    t = d['times']

    flat_all = factor.reshape(factor.shape[0], -1)
    lo = np.nanpercentile(flat_all, 1)
    hi = np.nanpercentile(flat_all, 99)
    edges = np.linspace(lo, hi, bins + 1)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    colours = plt.cm.viridis(np.linspace(0.15, 0.85, len(date_ranges)))

    for (lbl_a, lbl_b), (mjd_a, mjd_b), col in zip(date_ranges, mjd_ranges, colours):
        sel = (t >= mjd_a) & (t <= mjd_b)
        vals = factor[sel].ravel()
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue
        med = np.median(vals)
        ax.hist(vals, bins=edges, histtype='stepfilled', alpha=0.35,
                color=col, label=f'{lbl_a} – {lbl_b}  (N={vals.size}, med={med:.3f})')
        ax.hist(vals, bins=edges, histtype='step', color=col, lw=1.8)
        ax.axvline(med, color=col, ls=':', lw=1)

    ax.set_xlabel('Calibration factor  (model / measured)')
    ax.set_ylabel('Counts')
    ax.set_title(f'{source} calibration factor by epoch')
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    plt.show()


for src in SOURCES:
    plot_cal_histograms(src)